In [1]:
# Header for the notebook
from datetime import datetime
from IPython.display import display, Markdown

# Get the current date
title = "Data analysis project"
current_date = datetime.now().strftime("%d %B %Y, %H:%M:%S")
authors = "Jinwei Zhang (and Copilot)"

# Insert the date into the notebook
display(Markdown(f"# {title}"))
display(Markdown(f"{current_date}"))
display(Markdown(f"by {authors}"))

# Data analysis project

19 April 2026, 16:54:13

by Jinwei Zhang (and Copilot)

# Do surrogate DFA 
Surrogate DFA was used to test whether the observed long-range temporal structure of the weekly K series was greater than expected from randomly shuffled data.


In [4]:
# imports 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# The following magic command enables interactive matplotlib plots using ipywidgets.
# You may need to install 'ipywidgets' and 'ipympl' for this to work.
%matplotlib widget  

In [5]:
file_path = Path("data/cbtdata.xlsx")

df = pd.read_excel(file_path)
df.head()

,Semaine,C1A,BS1A,MS1A,K1A,C2A,BS2A,MS2A,K2A,C3A,...,MS4A,K4A,C5A,BS5A,MS5A,K5A,C6A,BS6A,MS6A,K6A
0,1,1.0,1.0,0.8,1.00,0.5,0.5,0.5,0.0,0.9,...,0.7,0.66,0.6,1.0,1.0,0.20,0.8,0.9,0.5,0.62
1,2,0.9,0.7,0.9,0.54,0.5,0.5,0.5,0.0,0.7,...,0.6,0.84,0.6,1.0,0.7,0.32,0.9,0.7,0.1,0.62
2,3,0.9,0.7,0.8,0.55,0.6,1.0,1.0,0.2,0.8,...,0.7,0.42,0.7,1.0,1.0,0.40,0.7,0.7,0.5,0.34
3,4,0.9,0.7,0.9,0.54,0.7,1.0,1.0,0.4,0.7,...,0.7,-0.02,0.4,1.0,1.0,-0.20,0.8,0.8,0.5,0.54
4,5,0.9,0.9,0.8,0.73,0.5,1.0,1.0,0.0,0.8,...,0.7,0.58,0.3,1.0,0.9,-0.33,0.8,0.8,0.5,0.54


In [ ]:
k_cols = ["Semaine", "K1A", "K2A", "K3A", "K4A", "K5A", "K6A"]

df_k = df[k_cols].copy()

df_long = df_k.melt(
    id_vars="Semaine",
    var_name="participant",
    value_name="K"
)

df_long = df_long.dropna(subset=["K"]).copy()
df_long = df_long.sort_values(["participant", "Semaine"]).reset_index(drop=True)

df_long.head()

In [ ]:
def compute_dfa_alpha(x, min_window=4, n_scales=15):
    """
    Compute DFA alpha from a 1D time series.
    Returns only the DFA scaling exponent alpha.
    """
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]

    if len(x) < 20:
        return np.nan

    y = np.cumsum(x - np.mean(x))
    n = len(y)

    max_window = n // 4
    if max_window <= min_window:
        return np.nan

    scales = np.unique(
        np.floor(
            np.logspace(np.log10(min_window), np.log10(max_window), num=n_scales)
        ).astype(int)
    )

    flucts = []

    for s in scales:
        n_segments = n // s
        if n_segments < 2:
            continue

        rms_values = []

        for i in range(n_segments):
            segment = y[i * s:(i + 1) * s]
            t = np.arange(s)

            coeffs = np.polyfit(t, segment, 1)
            trend = np.polyval(coeffs, t)
            detrended = segment - trend

            rms = np.sqrt(np.mean(detrended ** 2))
            rms_values.append(rms)

        flucts.append(np.mean(rms_values))

    flucts = np.array(flucts, dtype=float)
    valid = np.isfinite(flucts) & (flucts > 0)

    scales = scales[:len(flucts)][valid]
    flucts = flucts[valid]

    if len(scales) < 2:
        return np.nan

    log_scales = np.log(scales)
    log_flucts = np.log(flucts)

    slope, _ = np.polyfit(log_scales, log_flucts, 1)
    return float(slope)

In [ ]:
np.random.seed(123)
n_surrogates = 200

participants = df_long["participant"].unique()
results = []

for participant in participants:
    sub = df_long[df_long["participant"] == participant].sort_values("Semaine")
    x = sub["K"].dropna().values

    obs_alpha = compute_dfa_alpha(x)

    null_alpha = []
    for _ in range(n_surrogates):
        x_surr = np.random.permutation(x)
        alpha_surr = compute_dfa_alpha(x_surr)
        null_alpha.append(alpha_surr)

    null_alpha = np.array(null_alpha, dtype=float)
    null_alpha = null_alpha[np.isfinite(null_alpha)]

    results.append({
        "participant": participant,
        "obs_alpha": obs_alpha,
        "null_mean": np.mean(null_alpha) if len(null_alpha) > 0 else np.nan,
        "lower_2.5": np.quantile(null_alpha, 0.025) if len(null_alpha) > 0 else np.nan,
        "upper_97.5": np.quantile(null_alpha, 0.975) if len(null_alpha) > 0 else np.nan,
        "p_upper": np.mean(null_alpha >= obs_alpha) if len(null_alpha) > 0 else np.nan
    })

surrogate_dfa = pd.DataFrame(results)
surrogate_dfa

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

x_pos = np.arange(len(surrogate_dfa))
y = surrogate_dfa["obs_alpha"].values
yerr_lower = y - surrogate_dfa["lower_2.5"].values
yerr_upper = surrogate_dfa["upper_97.5"].values - y

ax.axhline(0.5, linestyle="--", linewidth=1)
ax.errorbar(
    x_pos,
    y,
    yerr=[yerr_lower, yerr_upper],
    fmt="o",
    capsize=4
)

ax.set_xticks(x_pos)
ax.set_xticklabels(surrogate_dfa["participant"].values)
ax.set_xlabel("Participant")
ax.set_ylabel("DFA alpha")
ax.set_title("Observed DFA alpha vs surrogate 95% interval")

plt.tight_layout()
plt.show()

In [ ]:
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

surrogate_dfa.to_csv(results_dir / "surrogate_dfa_python.csv", index=False)

print("surrogate DFA results saved.")

In [6]:
# https://nbconvert.readthedocs.io/en/latest/removing_cells.html

# https://github.com/msm1089/ipynbname/issues/17#issuecomment-1293269863


from traitlets.config import Config
from nbconvert.exporters import HTMLExporter
from nbconvert.preprocessors import TagRemovePreprocessor
from IPython import get_ipython


def get_notebook_name():
    """
    Get the current notebook name (without extension).
    """
    ip = get_ipython()
    path = None
    if "__vsc_ipynb_file__" in ip.user_ns:
        path = ip.user_ns["__vsc_ipynb_file__"]

    return path.split("/")[-1].split(".")[0]


# Get the notebook name
notebook_file_name = get_notebook_name()


# Setup config
c = Config()

# Configure tag removal - be sure to tag your cells to remove  using the
# words remove_cell to remove cells. You can also modify the code to use
# a different tag word
c.TagRemovePreprocessor.remove_cell_tags = ("remove",)
c.TagRemovePreprocessor.remove_all_outputs_tags = ("remove_output",)
c.TagRemovePreprocessor.remove_input_tags = ("hide",)
c.TagRemovePreprocessor.enabled = True
c.HTMLExporter.preprocessors = ["nbconvert.preprocessors.TagRemovePreprocessor"]

# ensure the graphics are included in the html
c.HTMLExporter.embed_images = True
# do not show the input code cells (distracts from the output)
c.HTMLExporter.exclude_output_prompt = True
c.HTMLExporter.exclude_input_prompt = True

# Configure the exporter
exporter = HTMLExporter(config=c)
exporter.register_preprocessor(TagRemovePreprocessor(config=c), True)


# run our exporter - returns a tuple - first element with html,
# second with notebook metadata
output = HTMLExporter(config=c).from_filename(notebook_file_name + ".ipynb")

# Write to output html file
with open(notebook_file_name + ".html", "w") as f:
    f.write(output[0])

# open the file with the operating system
import os

# if osx use open, if linux use xdg-open, if windows use start
try:
    if os.name == "posix":
        if os.uname().sysname == "Darwin":
            # macOS
            errorCode = os.system("open " + notebook_file_name + ".html")
        else:
            # Linux
            errorCode = os.system("xdg-open " + notebook_file_name + ".html")
    elif os.name == "nt":
        # Windows
        errorCode = os.system("start " + notebook_file_name + ".html")
    else:
        print("Unsupported OS")
except Exception as e:
    print("Error opening file: ", e)